# Gestión Cuantitativa de Riesgo de Crédito
 
#### Magíster en Finanzas Weekend - Universidad de Chile

##### Business Analytics 

**Franco Mansilla Ibáñez**

# 1. Instancias 

In [63]:
# Importar todas las librerías necesarias al inicio
import pandas as pd
import numpy as np
from scipy.stats import levene
from scipy import stats
from tabulate import tabulate
import plotly.graph_objects as go
import plotly.express as px
import plotly.figure_factory as ff
import statsmodels.api as sm
from mrmr import mrmr_classif
from sklearn.metrics import roc_curve, f1_score, roc_auc_score, confusion_matrix
from sklearn.base import is_classifier
from sklearn.ensemble import RandomForestClassifier

# 2. Base de Datos

In [2]:
file_path = '/Users/francomansilla/Library/CloudStorage/GoogleDrive-franco.andres.mansilla@gmail.com/Mi unidad/universidad de chile/Magíster en Finanzas WK/Business Analytics/Clase 2 - Riesgo Crédito/Python/'

df = pd.read_excel(file_path + 'Base de Datos Python.xlsx', sheet_name='Base de Datos')

print("Dimensión", df.shape)
df.head()

Dimensión (500, 10)


,default,edad,n_educ,a_emp,a_dir,ing_h,deu_ing,deu_tc,deu_o,muestra
0,0,29,2,6,10,65.0,18.4,1.990000,9.97,Entrenamiento
1,0,43,1,25,21,64.0,16.7,0.950000,9.74,Entrenamiento
2,1,32,2,11,6,75.0,23.3,5.495485,9.72,Entrenamiento
3,0,37,5,9,16,0.0,5.9,0.890000,9.56,Entrenamiento
4,0,45,6,22,24,91.0,11.7,1.260000,9.39,Entrenamiento


# 3. Análisis pre-eliminar

## 3.1. Discriminación Visual 

* Evalua si las variables que usamos para explicar la variable de intéres (`default`) son capaces de discriminar entre los buenos y malos pagadores, de forma visual.

In [3]:
# Dividir el DataFrame según los valores de "default"
grupo_default = df[df['default'] == 1]
grupo_no_default = df[df['default'] == 0]

# Crear los histogramas y asignar la configuración
fig = go.Figure()

fig.add_trace(go.Histogram(x=grupo_default['deu_ing'], nbinsx=20, opacity=0.1, name='Grupo Default', histnorm='percent',  marker_color='red'))
fig.add_trace(go.Histogram(x=grupo_no_default['deu_ing'], nbinsx=20, opacity=0.1, name='Grupo No Default', histnorm='percent',  marker_color='blue'))

# Configurar el layout para fondo blanco y títulos
fig.update_layout(
    barmode='overlay',
    title='Diferencias entre entre grupos',
    xaxis_title='deu_ing',
    yaxis_title='Frecuencia Relativa (%)',
    paper_bgcolor='rgba(255,255,255,1)',
    plot_bgcolor='rgba(255,255,255,1)',
    width=1000,  # Cambiar el ancho del gráfico
    height=400
)

# Mostrar el gráfico
fig.show()


## 3.2. Discriminación Estadística

* Se analiza si las variables explicativas utilizadas para modelar la variable de interés (`default`) permiten distinguir estadísticamente entre buenos y malos pagadores.
* Para comparar las medias de la variable de interés según el grupo (`default`), primero es necesario determinar si las varianzas de ambos grupos son iguales o diferentes. Para ello, se utiliza la prueba de varianza de `Levene`.

In [4]:
def sign_estadistica(df, var_grupo, alpha_varianza=0.05):
    rs_analisis = []

    cat_0 = df[var_grupo].unique()[0]
    cat_1 = df[var_grupo].unique()[1]
    
    # Solo columnas numéricas
    cols_numericas = df.select_dtypes(include=[np.number]).columns.tolist()
    for col in cols_numericas:
        if col != var_grupo:
            grupo_0 = df[df[var_grupo] == cat_0]
            grupo_1 = df[df[var_grupo] == cat_1]
            try:
                statistic, p_value_varianza = levene(grupo_0[col], grupo_1[col])
                if (p_value_varianza >= alpha_varianza):
                    t_stat, p_value = stats.ttest_ind(grupo_0[col], grupo_1[col], equal_var=True, nan_policy='omit')
                else:
                    t_stat, p_value = stats.ttest_ind(grupo_0[col], grupo_1[col], equal_var=False, nan_policy='omit')
                rs_analisis.append([col, p_value_varianza, np.abs(t_stat), p_value])
            except Exception as e:
                # Si hay error, lo ignora y sigue
                continue
    t_stats = [(fila[0], fila[2]) for fila in rs_analisis]

    resultado_ordenado = sorted(rs_analisis, key=lambda x: x[2], reverse=True)
    headers = ['Variable', 'P-value levene', 'Estadística t', 'P-value']
    return tabulate(resultado_ordenado, headers=headers, floatfmt=".4f"), t_stats

* Recordar el nivel de significancia de la prueba de `levene` para saber si se no se rechaza o se rechaza la hipótesis nula de la prueba de varianza, `alpha_varianza = 0.01`

In [5]:
sign_stats, _ = sign_estadistica(df, 'default', alpha_varianza=0.01)
print(sign_stats)

Variable      P-value levene    Estadística t    P-value
----------  ----------------  ---------------  ---------
deu_ing               0.0030           7.3148     0.0000
a_emp                 0.0085           6.4934     0.0000
deu_tc                0.0001           3.7391     0.0002
a_dir                 0.0687           2.7665     0.0059
ing_h                 0.3847           2.5400     0.0114
deu_o                 0.1196           2.3077     0.0214
edad                  0.2456           1.9856     0.0476
n_educ                0.6352           1.8083     0.0712


* La variable que mas discrmina los buenos y malos pagadores (`default = 1`) es la variable ratio `deuda_ingreso` con un `estadístico t = 7.3148`

# 4. Modelos de Regresión Logística

* Separamos la muestras de `Entrenamiento` y `Validación`

In [6]:
df_ent = df.loc[df['muestra'] == 'Entrenamiento']
df_val = df.loc[df['muestra'] == 'Validación']

print("Dimensión Entrenamiento", df_ent.shape)
print("Dimensión Validación", df_val.shape)

Dimensión Entrenamiento (350, 10)
Dimensión Validación (150, 10)


In [7]:
x_ent = df_ent.drop(columns=['default', 'muestra'])
y_ent = df_ent['default']

x_val = df_val.drop(columns=['default', 'muestra'])
y_val = df_val['default']

print("Dimensión X Entrenamiento", x_ent.shape)
print("Dimensión y Entrenamiento", y_ent.shape)
print("Dimensión X Validación", x_val.shape)
print("Dimensión y Validación", y_val.shape)

Dimensión X Entrenamiento (350, 8)
Dimensión y Entrenamiento (350,)
Dimensión X Validación (150, 8)
Dimensión y Validación (150,)


* Construcción de la prueba no-parámetrica de Kolmogotov-Smirnovs (KS)

In [8]:
# Cálculo de KS
def ks_statistic(y_true, y_pred_proba):
    fpr, tpr, _ = roc_curve(y_true, y_pred_proba)
    ks = np.max(tpr - fpr)
    return ks

## 4.1. Prueba de MrMR (Mínima Redundancia - Máxima Relevancia)

* https://feature-engine.trainindata.com/en/1.8.x/user_guide/selection/MRMR.html

In [47]:
def ks_vars(estimator, x_ent, y_ent, x_val, y_val, max_vars=2, graph=False, subtitulo=''):
    ks_ent = []
    ks_val = []
    variable_names = []

    for num_vars in range(1, max_vars + 1):
        selected_features = mrmr_classif(x_ent, y_ent, K=num_vars)
        variable_names.append(list(selected_features))

        X_ent_selected = x_ent[selected_features]
        X_val_selected = x_val[selected_features]

        # Para modelos de statsmodels, agrega una constante
        if not is_classifier(estimator):
            X_ent_selected = sm.add_constant(X_ent_selected)
            X_val_selected = sm.add_constant(X_val_selected)
            model = estimator(y_ent, X_ent_selected)
            fitted_model = model.fit()
            y_pred_proba_ent = fitted_model.predict(X_ent_selected)
            y_pred_proba_val = fitted_model.predict(X_val_selected)

        else:
            # Para clasificadores de sklearn
            estimator.fit(X_ent_selected, y_ent)
            y_pred_proba_ent = estimator.predict_proba(X_ent_selected)[:, 1]
            y_pred_proba_val = estimator.predict_proba(X_val_selected)[:, 1]

        ks_ent_stat = ks_statistic(y_ent, y_pred_proba_ent)
        ks_val_stat = ks_statistic(y_val, y_pred_proba_val)

        ks_ent.append(ks_ent_stat)
        ks_val.append(ks_val_stat)

    if graph:
        import plotly.graph_objects as go
        fig = go.Figure()
        fig.add_trace(go.Scatter(x=np.arange(1, max_vars + 1), y=ks_ent, mode='lines+markers', name='Entrenamiento', line=dict(color='black', dash='solid')))
        fig.add_trace(go.Scatter(x=np.arange(1, max_vars + 1), y=ks_val, mode='lines+markers', name='Validación', line=dict(color='black', dash='dash')))
        fig.update_layout(title='Selección de variables usando KS',
                          xaxis_title='Número de variables',
                          yaxis_title='Estadístico KS',
                          legend_title='Conjunto',
                          width=800, height=400,
                          template='simple_white',
                          title_text='Selección de Features usando Kolmogorov-Smirnov (KS)<br>'+subtitulo)
        fig.show()

    return ks_ent, ks_val, variable_names

In [10]:
# Regresión Logística
max_vars = len(x_ent.columns.tolist())
logit_model = sm.Logit
ks_ent_vars_cl, ks_val_vars_cl, var_names = ks_vars(logit_model, x_ent, y_ent, x_val, y_val, max_vars=max_vars, graph=True
                                                                    ,subtitulo='Regresión Logística')

100%|██████████| 1/1 [00:00<00:00, 466.19it/s]


Optimization terminated successfully.
         Current function value: 0.475789
         Iterations 6


100%|██████████| 2/2 [00:00<00:00, 13.82it/s]


Optimization terminated successfully.
         Current function value: 0.430364
         Iterations 7


100%|██████████| 3/3 [00:00<00:00, 13.71it/s]

...

Optimization terminated successfully.
         Current function value: 0.412811
         Iterations 7



100%|██████████| 7/7 [00:03<00:00,  1.83it/s]

Optimization terminated successfully.
         Current function value: 0.410132
         Iterations 7



100%|██████████| 8/8 [00:03<00:00,  2.26it/s]


Optimization terminated successfully.
         Current function value: 0.410041
         Iterations 7


## 4.2. Modelo final - Regresión Logística

In [38]:
# Asegúrate de que var_names está definido correctamente
selected_features = var_names[3]

# Añadiendo la constante al modelo
X_ent = sm.add_constant(x_ent[selected_features])
X_val = sm.add_constant(x_val[selected_features])

# Ajustando el modelo logístico
model = sm.Logit(y_ent, X_ent)
result = model.fit()

print(result.summary())

Optimization terminated successfully.
         Current function value: 0.416074
         Iterations 7
                           Logit Regression Results                           
Dep. Variable:                default   No. Observations:                  350
Model:                          Logit   Df Residuals:                      345
Method:                           MLE   Df Model:                            4
Date:                Mon, 15 Dec 2025   Pseudo R-squ.:                  0.2209
Time:                        21:37:38   Log-Likelihood:                -145.63
converged:                       True   LL-Null:                       -186.92
Covariance Type:            nonrobust   LLR p-value:                 4.944e-17
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
const         -1.4270      0.350     -4.074      0.000      -2.114      -0.740
deu_ing        0.1259      0.

### 4.2.1. Evaluación Metodologicia - Modelo final

Se calculan: 
* El `AUC` (Area Under the Curve) de la curva ROC (Receiver Operating Characteristic)  
* La `Matriz de Confusión`.  
* El `KS` (Kolmogorov-Smirnov)  
* La `Precisión`, `Recall` y `F1-Score`  

In [12]:
# Predicciones de probabilidad
y_pred_proba_ent = result.predict(X_ent)
y_pred_proba_val = result.predict(X_val)

# Umbral óptimo (por defecto 0.5, pero puedes optimizarlo)
threshold = 0.5
y_pred_ent = (y_pred_proba_ent >= threshold).astype(int)
y_pred_val = (y_pred_proba_val >= threshold).astype(int)

# F1-Score
f1_ent = f1_score(y_ent, y_pred_ent)
f1_val = f1_score(y_val, y_pred_val)

# KS
ks_ent = ks_statistic(y_ent, y_pred_proba_ent)
ks_val = ks_statistic(y_val, y_pred_proba_val)

# AUC
auc_ent = roc_auc_score(y_ent, y_pred_proba_ent)
auc_val = roc_auc_score(y_val, y_pred_proba_val)

print(f"F1-Score Entrenamiento: {f1_ent:.3f}")
print(f"F1-Score Validación: {f1_val:.3f}")
print(f"KS Entrenamiento: {ks_ent:.3f}")
print(f"KS Validación: {ks_val:.3f}")
print(f"AUC Entrenamiento: {auc_ent:.3f}")
print(f"AUC Validación: {auc_val:.3f}")

F1-Score Entrenamiento: 0.475
F1-Score Validación: 0.447
KS Entrenamiento: 0.499
KS Validación: 0.470
AUC Entrenamiento: 0.813
AUC Validación: 0.750


In [13]:
cm = confusion_matrix(y_ent, y_pred_ent)
labels = ['No Default', 'Default']
z = cm
x = labels
y = labels
fig = ff.create_annotated_heatmap(z, x=x, y=y, colorscale='Blues', showscale=True, reversescale=False)
fig.update_layout(title_text='Matriz de Confusión - Entrenamiento', xaxis_title='Predicción', yaxis_title='Real', width=500, height=400)
fig.show()

In [26]:
df_ent['confusion_group'] = np.select(
    [
        (y_ent == 1) & (y_pred_ent == 1),  # TP
        (y_ent == 0) & (y_pred_ent == 0),  # TN
        (y_ent == 0) & (y_pred_ent == 1),  # FP
        (y_ent == 1) & (y_pred_ent == 0)   # FN
    ],
    ['TP', 'TN', 'FP', 'FN']
)


* Discriminación de `TP`, `TN`, `FP` y `FN` a través de gráficas de densidad (KDE) por variables que califican en el modelo final.

In [ ]:
def plot_kde_by_confusion_group(
    df,
    features,
    group_col="confusion_group",
    order=("TN", "TP", "FP", "FN")
):
    for var in features:

        data = [
            df.loc[df[group_col] == g, var].dropna()
            for g in order
        ]

        fig = ff.create_distplot(
            hist_data=data,
            group_labels=list(order),
            show_hist=False,
            show_rug=False,
            curve_type="kde"
        )

        fig.update_layout(
            title=f"Distribución de {var} por tipo de predicción",
            xaxis_title=var,
            yaxis_title="Densidad",
            width=900,
            height=450,
            legend_title_text="Tipo de predicción"
        )

        fig.show()


* Variable que no califica en el modelo final (permite comparar)

In [ ]:
plot_kde_by_confusion_group(
    df=df_ent,
    features=['ing_h']
)

* Variables que si participan en el modelo final

In [42]:

plot_kde_by_confusion_group(
    df=df_ent,
    features=selected_features
)


### 4.2.2. Evaluación de Negocio - Modelo final

* **Paso 1.** De la muestra de Entrenamiento, se divide en 10 bucket la probabilidad estimada por el modelo final.
* **Paso 2.** Se grafica cada uno de los 10 buckets vs la tasa de default observada en cada bucket promediando la variable `default`.
* **Paso 3.** Los mismo puntos de corte de cada bucket de la muestra de Entrenamiento, se aplican a la muestra de Validación y se grafica cada uno de los 10 buckets vs la tasa de default observada en cada bucket promediando la variable `default` pero en la muestra de Validación.

In [59]:
def distribucion_score(df_train, df_val, n_segments=10, score_col='SCORE', target_col='default', plot_graph=False):
    import plotly.graph_objects as go
    # Crear intervalos y etiquetas para los scores usando quantiles en el DataFrame de entrenamiento
    bins = pd.qcut(df_train[score_col], q=n_segments, retbins=True, duplicates='drop')[1]
    labels = [f"{round(bins[i], 2)} - {round(bins[i+1], 2)}" for i in range(len(bins)-1)]

    # Asignar los intervalos a los DataFrames
    df_train = df_train.copy()
    df_val = df_val.copy()
    df_train['Perfil'] = pd.cut(df_train[score_col], bins=bins, labels=labels, include_lowest=True)
    df_val['Perfil'] = pd.cut(df_val[score_col], bins=bins, labels=labels, include_lowest=True)

    # Resumen para entrenamiento
    resumen_train = df_train.groupby('Perfil').agg(
        Total=('Perfil', 'size'),
        Promedio_Default=(target_col, 'mean')
    ).reset_index()

    # Resumen para validación
    resumen_val = df_val.groupby('Perfil').agg(
        Total=('Perfil', 'size'),
        Promedio_Default=(target_col, 'mean')
    ).reset_index()

    # Generar gráficos si plot_graph es True
    if plot_graph:
        # Gráfico para el conjunto de entrenamiento
        fig_train = go.Figure()
        fig_train.add_trace(go.Bar(x=resumen_train['Perfil'], y=resumen_train['Total'], name='Total - Train', marker_color='indigo', yaxis='y1'))
        fig_train.add_trace(go.Scatter(x=resumen_train['Perfil'], y=resumen_train['Promedio_Default'], name='Probabilidad de Default - Train', marker=dict(color='orange', symbol='circle'), mode='lines+markers', yaxis='y2'))
        fig_train.update_layout(
            title='Distribución de Score y Default por Perfil - Entrenamiento',
            xaxis=dict(title='Perfil'),
            yaxis=dict(title='Total de Observaciones - Train', titlefont=dict(color='indigo')),
            yaxis2=dict(title='Probabilidad de Default - Train', titlefont=dict(color='orange'), overlaying='y', side='right'),
            legend=dict(orientation='h', yanchor='bottom', y=1.1, xanchor='center', x=0.5),
            width=900, height=400,
            template='simple_white'
        )
        fig_train.show()

        # Gráfico para el conjunto de validación
        fig_val = go.Figure()
        fig_val.add_trace(go.Bar(x=resumen_val['Perfil'], y=resumen_val['Total'], name='Total - Validation', marker_color='teal', yaxis='y1'))
        fig_val.add_trace(go.Scatter(x=resumen_val['Perfil'], y=resumen_val['Promedio_Default'], name='Probabilidad de Default - Validation', marker=dict(color='red', symbol='circle'), mode='lines+markers', yaxis='y2'))
        fig_val.update_layout(
            title='Distribución de Score y Default por Perfil - Validación',
            xaxis=dict(title='Perfil'),
            yaxis=dict(title='Total de Observaciones - Validation', titlefont=dict(color='teal')),
            yaxis2=dict(title='Probabilidad de Default - Validation', titlefont=dict(color='red'), overlaying='y', side='right'),
            legend=dict(orientation='h', yanchor='bottom', y=1.1, xanchor='center', x=0.5),
            width=900, height=400,
            template='simple_white'
        )
        fig_val.show()

    return resumen_train, resumen_val

* Lo que buscamos en estas graficias que el modelo logre dismcriminar los mejores perfiles (menores bucket vs menores tasas de default observadas) a mayor probabilidad de incumplimiento observada, es decir, que la probabilidad de incumplimiento sea monotonicamente creciente a mayores niveles de buckets.

In [ ]:
# Uso de la función
df_ent['SCORE_logistica'] = y_pred_proba_ent
df_val['SCORE_logistica'] = y_pred_proba_val
dist_ent, dist_val = distribucion_score(df_ent, df_val, n_segments=10, score_col='SCORE_logistica', plot_graph=True)

Observar como quedan los nuevos perfiles agrupando bucket: 

* Bucket 1 = Perfil 1
* Bucket 2 = Perfil 2
* Bucket 3, 4 y 5 = Perfil 3
* Bucket 6, 7 y 8 = Perfil 4
* Bucket 9 = Perfil 5
* Bucket 10 = Perfil 6

# 5. Random Forest Classifier

* Aplicamos el mismo concepto, donde califican las mejores 4 variables dado que despues el KS en la muestra de validación se comienza a estabilizar.

In [51]:
# Random Forest
from sklearn.ensemble import RandomForestClassifier

max_vars = len(x_ent.columns.tolist())
rf_model = RandomForestClassifier(n_estimators=10, criterion = 'entropy', max_depth = 5, min_samples_split = 10, random_state=42)
ks_ent_vars_rf, ks_val_vars_rf, var_names_rf = ks_vars(rf_model, x_ent, y_ent, x_val, y_val, max_vars=max_vars, graph=True, subtitulo='Random Forest')

100%|██████████| 2/2 [00:00<00:00, 10.86it/s]

100%|██████████| 4/4 [00:01<00:00,  2.08it/s]

100%|██████████| 5/5 [00:01<00:00,  3.28it/s]

100%|██████████| 6/6 [00:02<00:00,  2.26it/s]

100%|██████████| 7/7 [00:03<00:00,  1.92it/s]

100%|██████████| 8/8 [00:03<00:00,  2.20it/s]



* Los algoritmos de Random Forest utilizan la importancia para clasificiar las variables de mayor a menor importancia dentro de la descriminación de la variable default.

In [ ]:
# Importancia de variables con Random Forest (barras verticales)
selected_features_rf = var_names_rf[3]  # Selecciona el mismo número de variables que en el ejemplo anterior

# Entrenamiento del modelo con las variables seleccionadas
rf_model_imp = RandomForestClassifier(n_estimators=10, criterion = 'entropy', max_depth = 5, min_samples_split = 10, random_state=42)
rf_model_imp.fit(x_ent[selected_features_rf], y_ent)

# Importancia de las variables
importancias = rf_model_imp.feature_importances_

df_importancias = pd.DataFrame({'Variable': selected_features_rf, 'Importancia': importancias})
df_importancias = df_importancias.sort_values(by='Importancia', ascending=False)
print(df_importancias)

# Gráfico de importancia (barras verticales)
fig = px.bar(df_importancias, x='Variable', y='Importancia', title='Importancia de Variables - Random Forest', color='Importancia', color_continuous_scale='Blues', orientation='v')
fig.update_layout(xaxis_title='Variable', yaxis_title='Importancia', bargap=0.2)
fig.show()

  Variable  Importancia
0  deu_ing     0.374053
1    a_emp     0.252246
3   deu_tc     0.207864
2    a_dir     0.165837


## 5.2. Evaluación Metodologicia - Modelo Random Forest

* Tiende a mejorar 3 puntos porcentuales el modelo random forest en la muestra de validación vs el modelo de regresión logística.

In [57]:
# Predicciones y métricas con Random Forest
y_pred_proba_ent_rf = rf_model_imp.predict_proba(x_ent[selected_features_rf])[:, 1]
y_pred_proba_val_rf = rf_model_imp.predict_proba(x_val[selected_features_rf])[:, 1]

# Umbral óptimo (por defecto 0.5, pero puedes optimizarlo)
threshold = 0.5
y_pred_ent_rf = (y_pred_proba_ent_rf >= threshold).astype(int)
y_pred_val_rf = (y_pred_proba_val_rf >= threshold).astype(int)

# F1-Score
f1_ent_rf = f1_score(y_ent, y_pred_ent_rf)
f1_val_rf = f1_score(y_val, y_pred_val_rf)

# KS
ks_ent_rf = ks_statistic(y_ent, y_pred_proba_ent_rf)
ks_val_rf = ks_statistic(y_val, y_pred_proba_val_rf)

# AUC
auc_ent_rf = roc_auc_score(y_ent, y_pred_proba_ent_rf)
auc_val_rf = roc_auc_score(y_val, y_pred_proba_val_rf)

print(f"F1-Score Entrenamiento: {f1_ent_rf:.3f}")
print(f"F1-Score Validación: {f1_val_rf:.3f}")
print(f"KS Entrenamiento: {ks_ent_rf:.3f}")
print(f"KS Validación: {ks_val_rf:.3f}")
print(f"AUC Entrenamiento: {auc_ent_rf:.3f}")
print(f"AUC Validación: {auc_val_rf:.3f}")

F1-Score Entrenamiento: 0.661
F1-Score Validación: 0.453
KS Entrenamiento: 0.742
KS Validación: 0.500
AUC Entrenamiento: 0.936
AUC Validación: 0.783


## 5.3. Evaluación de Negocio - Modelo Random Forest

* Tiende a ordenar mejor los perfiles de riesgo vs probabilidad de incumplimiento observada, siendo que tienen las mismas variables que califican en el modelo final de regresión logística.

In [60]:
# Uso de la función para Random Forest
df_ent['SCORE_rf'] = y_pred_proba_ent_rf
df_val['SCORE_rf'] = y_pred_proba_val_rf
dist_ent_rf, dist_val_rf = distribucion_score(df_ent, df_val, n_segments=10, score_col='SCORE_rf', plot_graph=True)